# 02 — Embeddings and search by hand

**Idea:** turn every law section and every question into a vector, then find the sections whose vectors point in the most similar direction to the question's vector. No search library, only NumPy.

This is a toy setting: 10 **dev** questions and 200 sections, with every correct section included on purpose. Scores here are **not** results; the real evaluation (500 questions, 5,127 sections) is in M4.

In [1]:
import os
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from thairag.data import load_corpus, load_questions
from thairag.splits import load_split

pd.set_option("display.max_colwidth", 120)
SEED = 42
BGE_M3_SHA = "5617a9f61b028005a4858fdac845db406aefb181"

W1125 13:59:45.460000 6327 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. Pick 10 dev questions and 200 sections
The other 190 sections come from the *same laws* as the 10 questions. Random sections from all 36 laws would be too easy to tell apart.

In [2]:
corpus = load_corpus()
dev = load_split("dev").merge(load_questions("train")[["question_id", "question"]], on="question_id")
questions = dev.sample(10, random_state=SEED).reset_index(drop=True)

is_correct = corpus["section_id"].isin(questions["section_id"])
same_law = corpus["law_name"].isin(questions["law_name"])
distractors = corpus[same_law & ~is_correct].sample(200 - is_correct.sum(), random_state=SEED)
sections = pd.concat([corpus[is_correct], distractors]).reset_index(drop=True)

print(len(sections), "sections from", sections["law_name"].nunique(), "laws")
questions[["question_id", "question", "law_name"]]

200 sections from 8 laws


,question_id,question,law_name
0,train-05581,ผู้ประกอบการธุรกิจสัญญาซื้อขายล่วงหน้า สามารถนำหลักทรัพย์ของลูกค้าที่โอนมาเพื่อใช้ประโยชน์อะไร,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546
1,train-00555,พนักงานเจ้าหน้าที่ของกรมพัฒนาพลังงานทดแทนและอนุรักษ์พลังงานมีหน้าที่อะไรบ้าง,พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535
2,train-05783,เมื่อไหร่ที่พระราชบัญญัติสมาคมการค้ามีผลบังคับใช้?,พระราชบัญญัติสมาคมการค้า พ.ศ. 2509
3,train-01820,ถ้าเจ้าหนี้ไม่ยอมทวงหนี้กับลูกหนี้ ผู้ค้ำประกันสามารถดำเนินการอย่างไรได้บ้าง,ประมวลกฎหมายแพ่งและพาณิชย์
4,train-00767,การยื่นแบบแสดงรายการเงินได้ ให้ยื่น ณ ​ที่ใด,พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514
5,train-06920,เมื่อผู้ประกอบธุรกิจไม่ได้มาชำระบัญชีหลังจากเลิกประกอบการแล้ว จะต้องส่งมอบสิ่งใดให้กับเจ้าหน้าที่บัญชีภายในเวลาเท่าใด,พระราชบัญญัติการบัญชี พ.ศ. 2543
6,train-06050,ใครที่สามารถทำอาชีพผู้ทำบัญชีได้,พระราชบัญญัติวิชาชีพบัญชี พ.ศ. 2547
7,train-01131,ถ้าคุณรู้ว่ามีคนเบียดบังทรัพย์ของบริษัทหลักทรัพย์ คุณควรทำอย่างไร,พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535
8,train-00522,เมื่อได้รับคำสั่งจากอธิบดีในกรณีที่มีเหตุอันสมควร โรงงานควบคุมต้องปฏิบัติตามภายในกี่วัน,พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535
9,train-07753,ใครบ้างที่สามารถชำระหนี้ได้นอกจากลูกหนี้,ประมวลกฎหมายแพ่งและพาณิชย์


## 2. Embed: text in, vector out
`encode` runs each text through BGE-M3 and returns one vector of 1,024 numbers. `normalize_embeddings=True` scales every vector to length 1.

In [3]:
model = SentenceTransformer("BAAI/bge-m3", revision=BGE_M3_SHA)
S = model.encode(sections["text"].tolist(), normalize_embeddings=True, batch_size=16)
Q = model.encode(questions["question"].tolist(), normalize_embeddings=True)
print("sections:", S.shape, "| questions:", Q.shape)
print("first 5 numbers of question 0:", Q[0, :5].round(3))
print("vector lengths:", np.linalg.norm(S, axis=1).min().round(4), "to", np.linalg.norm(S, axis=1).max().round(4))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

sections: (200, 1024) | questions: (10, 1024)
first 5 numbers of question 0: [-0.06   0.013 -0.02   0.033 -0.013]
vector lengths: 1.0 to 1.0


## 3. Cosine similarity by hand
Cosine = dot product divided by the two lengths. Because the lengths are already 1, the plain dot product gives the same number, and one matrix product compares every question with every section.

In [4]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = Q @ S.T  # 10 questions x 200 sections
print("shape:", scores.shape)
print("by hand:", round(float(cosine(Q[0], S[0])), 4), "| matrix:", round(float(scores[0, 0]), 4))

shape: (10, 200)
by hand: 0.4116 | matrix: 0.4116


## 4. Search = sort by similarity

In [5]:
def rank_of_correct(scores):
    """Rank (1 = best) of the correct section for each question. Ties count against us."""
    ids = sections["section_id"].to_numpy()
    ranks = []
    for row, correct in zip(scores, questions["section_id"]):
        s = row[ids == correct][0]
        ranks.append(int((row >= s).sum()))
    return np.array(ranks)


def top3(scores):
    rows = []
    for i, row in enumerate(scores):
        for rank, j in enumerate(np.argsort(-row, kind="stable")[:3], start=1):
            rows.append(
                {
                    "q": i,
                    "rank": rank,
                    "score": round(float(row[j]), 3),
                    "correct": "YES" if sections["section_id"][j] == questions["section_id"][i] else "",
                    "law": sections["law_name"][j][:40],
                    "section": sections["section_num"][j],
                    "text": " ".join(sections["text"][j].split())[:110],
                }
            )
    return pd.DataFrame(rows).set_index(["q", "rank"])


dense_rank = rank_of_correct(scores)
print("rank of the correct section:", dense_rank.tolist())
top3(scores)

rank of the correct section: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


score correct                                       law section  \
q rank                                                                    
0 1     0.691     YES  พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2      36   
  2     0.679          พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2      32   
  3     0.605          พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์   223/3   
1 1     0.584     YES  พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังง      47   
  2     0.507          พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังง      59   
  3     0.494          พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังง      44   
2 1     0.682     YES        พระราชบัญญัติสมาคมการค้า พ.ศ. 2509       2   
  2     0.624                พระราชบัญญัติสมาคมการค้า พ.ศ. 2509      10   
  3     0.587                พระราชบัญญัติสมาคมการค้า พ.ศ. 2509      34   
3 1     0.689     YES                ประมวลกฎหมายแพ่งและพาณิชย์     688   
  2     0.649                        ประมวลกฎหมายแพ่งและพาณิชย์    1742   
  3     0.632                        ประมวลกฎหมายแพ่งและพาณิชย์     697   
4 1     0.693     YES  พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ.       40   
  2     0.527                   พระราชบัญญัติการบัญชี พ.ศ. 2543       8   
  3     0.503          พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์      89   
5 1     0.770     YES           พระราชบัญญัติการบัญชี พ.ศ. 2543      17   
  2     0.654                        ประมวลกฎหมายแพ่งและพาณิชย์    1270   
  3     0.626                   พระราชบัญญัติการบัญชี พ.ศ. 2543      15   
6 1     0.674     YES       พระราชบัญญัติวิชาชีพบัญชี พ.ศ. 2547      44   
  2     0.616                   พระราชบัญญัติการบัญชี พ.ศ. 2543       8   
  3     0.570                   พระราชบัญญัติการบัญชี พ.ศ. 2543      17   
7 1     0.641     YES  พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์     308   
  2     0.620          พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์     128   
  3     0.619          พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์     244   
8 1     0.656     YES  พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังง      10   
  2     0.551          พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังง      47   
  3     0.547          พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์     128   
9 1     0.724     YES                ประมวลกฎหมายแพ่งและพาณิชย์     314   
  2     0.598                        ประมวลกฎหมายแพ่งและพาณิชย์     688   
  3     0.595                        ประมวลกฎหมายแพ่งและพาณิชย์     320   

                                                                                                                  text  
q rank                                                                                                                  
0 1     พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 36 ในกรณีที่ลูกค้าผิดนัดชำระหนี้อันเนื่องมาจากการซื้อขายสัญญ  
  2     พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 32 ในส่วนนี้ (1) “ลูกค้า” หมายความว่า บุคคลซึ่งใช้บริการจากผ  
  3     พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535 มาตรา 223/3 ในกรณีที่สำนักหักบัญชีได้รับมาหรือมีไว้ซึ่งทรัพ  
1 1     พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535 มาตรา 47 เพื่อปฏิบัติการให้เป็นไปตามพระราชบัญญัตินี้ ให้พ  
  2     พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535 มาตรา 59 ผู้ใดขัดขวางหรือไม่อำนวยความสะดวกแก่พนักงานเจ้าห  
  3     พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535 มาตรา 44 เมื่อมีกรณีที่ต้องดำเนินการเรียกเก็บค่าธรรมเนียม  
2 1     พระราชบัญญัติสมาคมการค้า พ.ศ. 2509 มาตรา 2 พระราชบัญญัตินี้ให้ใช้บังคับตั้งแต่วันถัดจากวันประกาศในราชกิจจานุเบ  
  2     พระราชบัญญัติสมาคมการค้า พ.ศ. 2509 มาตรา 10 เมื่อนายทะเบียนได้รับคำขออนุญาตและพิจารณาแล้วเห็นว่าข้อบังคับไม่ขั  
  3     พระราชบัญญัติสมาคมการค้า พ.ศ. 2509 มาตรา 34 เมื่อมีเหตุอันควรสงสัยว่าสมาคมการค้าใดจะดำเนินการไม่ชอบด้วยกฎหมาย   
3 1     ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 688 เมื่อเจ้าหนี้ทวงให้ผู้ค้ำประกันชำระหนี้ ผู้ค้ำประกันจะขอให้เรียกลูกหนี้ชำ  
  2     ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1742 ถ้าในการชำระหนี้ซึ่งค้างชำระอยู่แก่ตน เจ้าหนี้คนใดคนหนึ่งได้รับตั้งในระห  
  3     ประมวลกฎหมายแพ่งและพาณิชย์

What does a score mean? Compare the correct section's score with the scores of all the other sections. Only the *order* matters: there is no fixed "good" value.

In [6]:
ids = sections["section_id"].to_numpy()
pd.DataFrame(
    {
        "correct": [row[ids == c][0] for row, c in zip(scores, questions["section_id"])],
        "best wrong": [row[ids != c].max() for row, c in zip(scores, questions["section_id"])],
        "median wrong": [np.median(row[ids != c]) for row, c in zip(scores, questions["section_id"])],
    }
).round(3)

,correct,best wrong,median wrong
0,0.691,0.679,0.462
1,0.584,0.507,0.317
2,0.682,0.624,0.358
3,0.689,0.649,0.435
4,0.693,0.527,0.396
5,0.770,0.654,0.456
6,0.674,0.616,0.388
7,0.641,0.620,0.433
8,0.656,0.551,0.369
9,0.724,0.598,0.423


## 5. Keyword search, the naive way: split on spaces
Score = how many distinct "words" the question and the section share. In English, splitting on spaces roughly gives words. Thai doesn't put spaces between words, so look at what comes out.

In [7]:
def split_spaces(text):
    return text.split()

q0 = questions["question"][0]
print(len(split_spaces(q0)), "pieces:", split_spaces(q0))
print("English:", split_spaces("When must the buyer pay the price?"))

2 pieces: ['ผู้ประกอบการธุรกิจสัญญาซื้อขายล่วงหน้า', 'สามารถนำหลักทรัพย์ของลูกค้าที่โอนมาเพื่อใช้ประโยชน์อะไร']
English: ['When', 'must', 'the', 'buyer', 'pay', 'the', 'price?']


In [8]:
def keyword_scores(tokenise):
    sec = [set(tokenise(t)) for t in sections["text"]]
    qs = [set(tokenise(t)) for t in questions["question"]]
    return np.array([[len(q & s) for s in sec] for q in qs])


space_scores = keyword_scores(split_spaces)
space_rank = rank_of_correct(space_scores)
print("questions where the correct section shares no piece at all:",
      sum(row[ids == c][0] == 0 for row, c in zip(space_scores, questions["section_id"])), "of 10")
print("rank of the correct section (ties count against us):", space_rank.tolist())

questions where the correct section shares no piece at all: 9 of 10
rank of the correct section (ties count against us): [200, 200, 200, 200, 1, 200, 200, 200, 200, 200]


## 6. The same keyword search with a Thai word tokeniser
`pythainlp` finds word boundaries using a dictionary. Nothing else changes: same overlap count, same sections.

In [9]:
from pythainlp.tokenize import word_tokenize


def split_thai(text):
    return [w for w in word_tokenize(text, engine="newmm", keep_whitespace=False)]

print(split_thai(q0))
thai_rank = rank_of_correct(keyword_scores(split_thai))
print("rank of the correct section:", thai_rank.tolist())

['ผู้ประกอบการ', 'ธุรกิจ', 'สัญญาซื้อขาย', 'ล่วงหน้า', 'สามารถ', 'นำ', 'หลักทรัพย์', 'ของ', 'ลูกค้า', 'ที่', 'โอน', 'มา', 'เพื่อ', 'ใช้ประโยชน์', 'อะไร']
rank of the correct section: [1, 1, 47, 51, 1, 1, 74, 58, 1, 12]


## 7. Side by side
Ranks out of 200; 1 is best. A section tied with others gets the worst rank of its tie.

In [10]:
result = pd.DataFrame(
    {"BGE-M3": dense_rank, "split on spaces": space_rank, "Thai tokeniser": thai_rank},
    index=questions["question_id"],
)
summary = pd.DataFrame(
    {"in top 1": (result == 1).sum(), "in top 3": (result <= 3).sum(), "median rank": result.median()}
)
display(result)
summary

,BGE-M3,split on spaces,Thai tokeniser
question_id,,,
train-05581,1,200,1
train-00555,1,200,1
train-05783,1,200,47
train-01820,1,200,51
train-00767,1,1,1
train-06920,1,200,1
train-06050,1,200,74
train-01131,1,200,58
train-00522,1,200,1


,in top 1,in top 3,median rank
BGE-M3,10,10,1.0
split on spaces,1,1,200.0
Thai tokeniser,5,5,6.5


## 8. Why did Thai keyword search still miss some?
For each miss, compare the correct section with the section that won. Look at the length and at *which* words matched.

In [11]:
tokens = [set(split_thai(t)) for t in sections["text"]]
rows = []
for i in np.where(thai_rank > 1)[0]:
    q = set(split_thai(questions["question"][i]))
    c = int(np.where(ids == questions["section_id"][i])[0][0])
    w = int(np.argmax([len(q & t) for t in tokens]))
    for name, j in (("correct", c), ("winner", w)):
        rows.append({"q": i, "which": name, "words in section": len(tokens[j]), "shared": len(q & tokens[j]),
                     "shared words": " ".join(sorted(q & tokens[j]))})
pd.DataFrame(rows).set_index(["q", "which"])

words in section  shared                       shared words
q which                                                               
2 correct                18       2          พระราชบัญญัติ สมาคมการค้า
  winner                 54       3      ที่ พระราชบัญญัติ สมาคมการค้า
3 correct                35       3      ผู้ค้ำประกัน ลูกหนี้ เจ้าหนี้
  winner                 53       5  ถ้า ผู้ค้ำประกัน เจ้าหนี้ ได้ ไม่
6 correct                33       2                        บัญชี ผู้ทำ
  winner                119       5            ทำ ที่ บัญชี สามารถ ได้
7 correct                48       3                ของ ทรัพย์ เบียดบัง
  winner                122       6               ของ คน ทำ มี รู้ ว่า
9 correct                41       3               ชำระหนี้ ลูกหนี้ ได้
  winner                 71       4            ชำระหนี้ ที่ สามารถ ได้

- Every winner is a **longer** section than the correct one: more words, more chances to overlap by luck.
- Winners collect **common words** (ที่, ได้, มี, ทำ) that say nothing about the topic. Counting treats them the same as เบียดบัง (embezzle).
- Some questions use **different words** for the same thing: the question says ใคร (who) and อาชีพ (job), the section says ผู้ใด and วิชาชีพ.

BM25 (M4) is built to reduce the first two. The third needs a method that compares meaning, such as embeddings.

**Limits of this toy:** 10 questions, correct sections guaranteed present, 200 of 5,127 sections; and 7 of the 10 questions come from the part of train whose labels were picked from BGE-M3's own results (PLAN §12).